# 📑 Hoist EOM - Requests Report
 
## 🔗 Links
- **[BRD EOM Requests - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
  
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket - TBD]()**  
  A summary of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()


In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

##### Views

In [0]:
migrationDate = spark.sql(f"""
          SELECT case when PortfolioName = 'Mercury' then '2019-01-01'
                else min(cast(AssignmentDate as date)) end AS migrationDate,
          PortfolioName
          FROM `{env}-cdh`.primary.portfolio
          WHERE PortfolioName in {PortfolioName}
          GROUP BY PortfolioName
      """)
migrationDate.createOrReplaceTempView("migrationDate")

In [0]:
hoist_RequestTypeMapping_vw = spark.sql(f"""

SELECT 'Debt Certificate'							AS Name, 1 AS Index UNION
SELECT 'Premises visit'								AS Name, 2 AS Index UNION
SELECT 'Legalization of heirs'						AS Name, 3 AS Index UNION  --'Deceased announcement'
SELECT 'Health problems issues announcement'		AS Name, 4 AS Index UNION
SELECT 'Unemployment difficulties announcement'		AS Name, 5 AS Index UNION
SELECT 'Settlement Request'							AS Name, 6 AS Index UNION
SELECT 'Written Complaint'							AS Name, 7 AS Index UNION
SELECT 'Verbal Complaint'							AS Name, 8 AS Index UNION
SELECT 'Inbound for Redial'							AS Name, 9 AS Index UNION
SELECT 'Inform for Proxy'							AS Name, 10 AS Index UNION
SELECT 'Paid off Certificate'						AS Name, 11 AS Index UNION
SELECT 'Retrieve Legal Documentation'				AS Name, 12 AS Index UNION
SELECT 'Other Legal related matter'					AS Name, 13 AS Index UNION
SELECT 'Code of Conduct'							AS Name, 14 AS Index UNION
SELECT 'Other'										AS Name, 15 AS Index UNION
SELECT 'Retrieve Recorded Calls'					AS Name, 16 AS Index UNION
SELECT 'Demographic Info'							AS Name, 17 AS Index UNION
SELECT 'Pro migration Statements'					AS Name, 18 AS Index UNION  --'Retrieve Documentation'
SELECT 'Related to L.3869 account'				    AS Name, 19 AS Index UNION  --'Related to L.3869 payment plan'
SELECT 'Data protection'							AS Name, 20 AS Index UNION
SELECT 'Regulatory'									AS Name, 21 AS Index UNION
SELECT 'Payments Certificate'						AS Name, 22 AS Index UNION
SELECT 'Copy of Contract'							AS Name, 23 AS Index UNION
SELECT 'Service L.3869 certificate'					AS Name, 24 AS Index UNION
SELECT 'Payment Investigation'						AS Name, 25 AS Index UNION
SELECT 'Dispute request'							AS Name, 26 AS Index 
""")

hoist_RequestTypeMapping_vw.createOrReplaceTempView("hoist_RequestTypeMapping_vw")

In [0]:
NumberofRequestsMonthly_rnk = spark.sql(f"""
SELECT portfolioname, RequestCode, RequestType, SUM(ExposureBalance) ExposureBalance 
FROM (
    select distinct portfolioname,
            RequestCode, 
            RequestType,
            ExposureBalance,
            ExposureKey, 
            rank()over(partition by RequestCode order by  CreatedDate desc) rnk 
    from `{env}-cdh`.primary.partyrequest  
    where PortfolioName in {PortfolioName}
    AND LAST_DAY(CAST(CreatedDate AS DATE))=LAST_DAY('{run_date}')
    and RequestCommentStatus<>'Wrong Insertion'  
)
WHERE rnk=1
GROUP BY RequestCode, RequestType, portfolioname
""")
NumberofRequestsMonthly_rnk.createOrReplaceTempView("NumberofRequestsMonthly_rnk")   

In [0]:
NumberofRequestsMonthly = spark.sql(f"""
SELECT 
			RequestType				        					AS RequestType
        ,   portfolioname				
		,	CAST(COUNT(RequestCode) AS INT)     				AS NumberofRequests
		,	CAST(SUM(ExposureBalance) AS DECIMAL(18,2))			AS GBV
FROM NumberofRequestsMonthly_rnk	
GROUP BY RequestType, portfolioname
""")
NumberofRequestsMonthly.createOrReplaceTempView("NumberofRequestsMonthly")          

In [0]:
partyrequest = spark.sql(f"""
SELECT DISTINCT
 	  		ea.RequestCode  
 	  	,	RequestType 
        ,   ea.portfolioname
 	  	,	ea.ExposureBalance 
        ,   ea.RequestCommentStatusCode
 	  	,	RANK() OVER  (PARTITION BY ea.RequestCode ORDER by CreatedDate desc) rnk 
 	  	,   ea.CreatedDate  
 	  	,   ea.RequestCommentStatus  
FROM `{env}-cdh`.primary.partyrequest ea
LEFT JOIN migrationDate as md on md.portfolioname = ea.portfolioname
WHERE CAST(ea.CreatedDate AS DATE) <= '{run_date}'
 	AND CAST(ea.CreatedDate AS DATE) > md.migrationDate
 	AND ea.RequestCommentStatus <> 'Wrong Insertion'    
    AND ea.PortfolioName in {PortfolioName} 
        """)
partyrequest.createOrReplaceTempView("partyrequest")                      

In [0]:
partyrequest_1 = spark.sql(f"""
SELECT DISTINCT
 			ea.RequestCode  
 		,	ea.RequestType 
        ,   ea.portfolioname
 		,	SUM(ea.ExposureBalance) ExposureBalance
        ,   ea.RequestCommentStatusCode
 		,   ea.CreatedDate  
 		,   ea.RequestCommentStatus  
FROM partyrequest ea
WHERE rnk=1
GROUP BY    ea.RequestCode  
 		,	ea.RequestType 
        ,   ea.portfolioname
        ,   ea.RequestCommentStatusCode
 		,   ea.CreatedDate  
 		,   ea.RequestCommentStatus     
        """)
partyrequest_1.createOrReplaceTempView("partyrequest_1")                      

In [0]:
NumberofRequestsCum_rnk = spark.sql(f"""
  SELECT DISTINCT
  		ea.RequestCode  
  	,	RequestType 
    ,   ea.portfolioname
  	,	ea.ExposureBalance RequestedLoansBalance
  	,	CASE WHEN ea.RequestCommentStatusCode in ('1','2','3','4','10','11','12','13','14','15','16')  THEN 1 END AS NumberofPending
  	,	CASE WHEN ea.RequestCommentStatusCode in ('5','9') THEN 1 END AS NumberofPositive
  	,	CASE WHEN ea.RequestCommentStatusCode in ('6') THEN 1 END AS NumberofNegative
  	,	CASE WHEN ea.RequestCommentStatusCode in ('7') THEN 1 END AS NumberofCancelled								
  	,   CASE WHEN ea.RequestCommentStatusCode in ('1','2','3','4','10','11','12','13','14','15','16') THEN ea.RequestCode END AS Pending
  	,   CASE WHEN ea.RequestCommentStatusCode='1' THEN cast (datediff('{run_date}',sub.CreatedDate) AS FLOAT)
             WHEN ea.RequestCommentStatusCode in ('2','3','4','10','11','12','13','14','15','16') THEN cast (datediff(ea.CreatedDate ,sub.CreatedDate) AS FLOAT)
  		     END AS AvgofPending
  	,   CASE WHEN ea.RequestCommentStatusCode in ('5','9') THEN ea.RequestCode
  		     END AS Positive
  	,   CASE WHEN ea.RequestCommentStatusCode in ('5','9') THEN cast(datediff(ea.CreatedDate,sub.CreatedDate ) AS FLOAT)
  		     END AS AvgofPositive
  	,   CASE WHEN ea.RequestCommentStatusCode in ('6') THEN  cast(datediff(ea.CreatedDate,sub.CreatedDate ) AS FLOAT)
  			 END AS AvgofNegative
  	,   CASE WHEN ea.RequestCommentStatusCode in ('6') THEN ea.RequestCode
  			 END AS Negative
  	,   CASE WHEN ea.RequestCommentStatusCode in ('7') THEN  cast(datediff(ea.CreatedDate,sub.CreatedDate ) AS FLOAT)
  			 END AS AvgofCancelled
  	,   CASE WHEN ea.RequestCommentStatusCode in ('7') THEN ea.RequestCode
  			END AS Cancelled
  	,	RANK() OVER  (PARTITION BY ea.RequestCode ORDER BY ea.CreatedDate DESC) AS R
  	, sub.CreatedDate AS subdate
  	, ea.CreatedDate  AS eventdate
  	, ea.RequestCommentStatus   
   	FROM partyrequest_1 ea
  	LEFT JOIN  (
               SELECT DISTINCT RequestCode,PortfolioName, MIN(CreatedDate) AS CreatedDate
   			   FROM `{env}-cdh`.primary.partyrequest                    
   			   WHERE CreatedDate<='{run_date}' AND PortfolioName in {PortfolioName}
                 GROUP BY RequestCode, PortfolioName
			   )  sub on sub.requestcode=ea.RequestCode and ea.PortfolioName=sub.PortfolioName

        """)
NumberofRequestsCum_rnk.createOrReplaceTempView("NumberofRequestsCum_rnk")      

In [0]:
NumberofRequestsCum_rnk_1 = spark.sql(f"""
SELECT  	RequestCode  
 	  	,	RequestType 
        ,   portfolioname
 	  	,	SUM(RequestedLoansBalance) RequestedLoansBalance
 	  	,	NumberofPending
 	  	,	NumberofPositive
 	  	,	NumberofNegative
 	  	,	NumberofCancelled								
 	  	,   Pending
 	  	,   AvgofPending
 	  	,   Positive
 	  	,   AvgofPositive
 	  	,   AvgofNegative
 	  	,   Negative
 	  	,   AvgofCancelled
 	  	,   Cancelled
 	  	,   subdate
 	  	,   eventdate
 	  	,   RequestCommentStatus   
FROM NumberofRequestsCum_rnk
WHERE R=1
GROUP BY	RequestCode  
 		,	RequestType 
        ,   portfolioname
 		,	NumberofPending
 		,	NumberofPositive
 		,	NumberofNegative
 		,	NumberofCancelled								
 		,   Pending
 		,   AvgofPending
 		,   Positive
 		,   AvgofPositive
 		,   AvgofNegative
 		,   Negative
 		,   AvgofCancelled
 		,   Cancelled
 		,   subdate
 		,   eventdate
 		,   RequestCommentStatus
        """)
NumberofRequestsCum_rnk_1.createOrReplaceTempView("NumberofRequestsCum_rnk_1")                      

In [0]:
NumberofRequestsCum = spark.sql(f"""

SELECT 
    RequestType										    AS RequestType
,   portfolioname
,	CAST(COUNT(NumberofPending) AS INT)					AS NumberofPending
,	CAST(COUNT(NumberofPositive)  AS INT)				AS NumberofPositive
,	CAST(COUNT(NumberofNegative) AS INT)				AS NumberofNegative
,	CAST(COUNT(NumberofCancelled) AS INT)				AS NumberofCancelled
,	ROUND(SUM(AvgofPending)/COUNT(Pending),0)			AS AvgofPending
,	ROUND(SUM(AvgofPositive)/COUNT(Positive),0)			AS AvgofPositive
,	ROUND(SUM(AvgofNegative)/COUNT(Negative),0)			AS AvgofNegative
,	ROUND(SUM(AvgofCancelled)/COUNT(Cancelled),0)		AS AvgofCancelled
FROM NumberofRequestsCum_rnk_1      
GROUP BY RequestType, portfolioname

""")
NumberofRequestsCum.createOrReplaceTempView("NumberofRequestsCum")      

In [0]:
Final = spark.sql(f"""
SELECT	
			CAST('{run_date}' AS DATE) 	AS BusinessDate
		,	ea.Name	                	AS RequestType
        ,   CASE WHEN cum.RequestType IS NOT NULL THEN cum.portfolioname WHEN monthly.RequestType IS NOT NULL THEN monthly.portfolioname END  AS portfolioname
		,	monthly.NumberofRequests
		,	monthly.GBV
		,	cum.AvgofPending
		,	cum.AvgofPositive
		,	cum.AvgofNegative
		,	cum.AvgofCancelled
		,	cum.NumberofPending
		,	cum.NumberofPositive
		,	cum.NumberofNegative
		,	cum.NumberofCancelled
FROM hoist_RequestTypeMapping_vw   ea           
LEFT JOIN NumberofRequestsCum              cum         ON cum.RequestType = ea.Name
LEFT JOIN NumberofRequestsMonthly          monthly     ON monthly.RequestType = ea.Name AND monthly.PortfolioName=cum.PortfolioName
WHERE CASE WHEN cum.RequestType IS NOT NULL THEN cum.portfolioname WHEN monthly.RequestType IS NOT NULL THEN monthly.portfolioname END IS NOT NULL
""")
Final.createOrReplaceTempView("Final")            

In [0]:
target_table_path_requests = f"`{env}-cdh-delivery`.investorreports.hoist_Requests"

spark.sql(f"""DELETE FROM {target_table_path_requests} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_requests) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_requests} SYNC METADATA")